In [1]:
%load_ext autoreload
%autoreload 2

In [17]:
import torch
from torch import nn
from torchrl.modules import ProbabilisticActor, ValueOperator, TanhNormal
from torchrl.envs.libs.dm_control import DMControlEnv
from tensordict.nn import TensorDictModule

from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE

from torchrl.collectors import SyncDataCollector
from torch.optim import Adam

from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from tensordict.nn.distributions import NormalParamExtractor

In [3]:
# Environment setup
env = DMControlEnv("dog", "run", from_pixels=False)

In [13]:
check_env_specs(env)

2026-02-17 17:01:37,751 [torchrl][INFO]    check_env_specs succeeded! [END]


In [15]:
device = (
    torch.device(0)
    if torch.mps.is_available()
    else torch.device("cpu")
)
num_cells = 223  # number of cells in each layer i.e. output dim.
lr = 3e-4
max_grad_norm = 1.0

In [18]:
# Actor: Outputs mean and standard deviation for the Normal distribution
actor_net = nn.Sequential(
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(2 * env.action_spec.shape[-1], device=device),
    NormalParamExtractor(),
)

policy_module = TensorDictModule(
    actor_net, in_keys=["observation"], out_keys=["loc", "scale"]
)

policy_module = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec,
    in_keys=["loc", "scale"],
    distribution_class=TanhNormal,
    distribution_kwargs={
        "low": env.action_spec_unbatched.space.low,
        "high": env.action_spec_unbatched.space.high,
    },
    return_log_prob=True,
    # we'll need the log-prob for the numerator of the importance weights
)

# Critic: Estimates the state value (V-function)
value_net = nn.Sequential(
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(1, device=device),
)

value_module = ValueOperator(
    module=value_net,
    in_keys=["observation"],
)

In [19]:
sub_batch_size = 64  # cardinality of the sub-samples gathered from the current data in the inner loop
num_epochs = 10  # optimization steps per batch of data collected
clip_epsilon = (
    0.2  # clip value for PPO loss: see the equation in the intro for more context.
)
gamma = 0.99
lmbda = 0.95
entropy_eps = 1e-4

In [20]:
frames_per_batch = 1000
# For a complete training, bring the number of frames up to 1M
total_frames = 10_000

In [21]:
advantage_module = GAE(
    gamma=0.99, lmbda=0.95, value_network=value_module, average_gae=True
)
# PPO Loss module
loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=clip_epsilon,
    entropy_bonus=bool(entropy_eps),
    entropy_coeff=entropy_eps,
    # these keys match by default but we set this for completeness
    critic_coeff=1.0,
    loss_critic_type="smooth_l1",
)

optim = torch.optim.Adam(loss_module.parameters(), lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim, total_frames // frames_per_batch, 0.0
)

In [22]:
collector = SyncDataCollector(env, policy_module, frames_per_batch=2048)
optim = Adam(loss_module.parameters(), lr=3e-4)

replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=frames_per_batch),
    sampler=SamplerWithoutReplacement(),
)

KeyError: "Some tensors that are necessary for the module call may not have not been found in the input tensordict: the following inputs are None: {'observation'}."

In [ ]:
for i, data in enumerate(collector):
    # 1. Compute advantages
    with torch.no_grad():
        advantage_module(data)
    
    # 2. Optimization loop (multiple epochs per batch)
    for _ in range(10): 
        loss_vals = loss_module(data)
        loss_total = loss_vals["loss_objective"] + loss_vals["loss_critic"] + loss_vals["loss_entropy"]
        
        loss_total.backward()
        optim.step()
        optim.zero_grad()
    
    print(f"Iteration {i}: Reward = {data['next', 'reward'].mean().item()}")